In [ ]:
import numpy as np
import pandas as pd
import pickle
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense
from tensorflow.keras.callbacks import EarlyStopping

In [ ]:
df = pd.read_csv("/content/household_power_consumption.txt", sep=";", na_values="?", low_memory=False)
df.head()

,Date,Time,Global_active_power,Global_reactive_power,Voltage,Global_intensity,Sub_metering_1,Sub_metering_2,Sub_metering_3
0,16/12/2006,17:24:00,4.216,0.418,234.84,18.4,0.0,1.0,17.0
1,16/12/2006,17:25:00,5.360,0.436,233.63,23.0,0.0,1.0,16.0
2,16/12/2006,17:26:00,5.374,0.498,233.29,23.0,0.0,2.0,17.0
3,16/12/2006,17:27:00,5.388,0.502,233.74,23.0,0.0,1.0,17.0
4,16/12/2006,17:28:00,3.666,0.528,235.68,15.8,0.0,1.0,17.0


In [ ]:
print("Dataset Shape:")
print(df.shape)

print("Columns:")
print(df.columns.tolist())

print("Missing Values:")
print(df.isnull().sum())

Dataset Shape:
(2075259, 9)
Columns:
['Date', 'Time', 'Global_active_power', 'Global_reactive_power', 'Voltage', 'Global_intensity', 'Sub_metering_1', 'Sub_metering_2', 'Sub_metering_3']
Missing Values:
Date                         0
Time                         0
Global_active_power      25979
Global_reactive_power    25979
Voltage                  25979
Global_intensity         25979
Sub_metering_1           25979
Sub_metering_2           25979
Sub_metering_3           25979
dtype: int64


In [ ]:
df["Datetime"] = pd.to_datetime(df["Date"] + " " + df["Time"], dayfirst=True)
df.head()

,Date,Time,Global_active_power,Global_reactive_power,Voltage,Global_intensity,Sub_metering_1,Sub_metering_2,Sub_metering_3,Datetime
0,16/12/2006,17:24:00,4.216,0.418,234.84,18.4,0.0,1.0,17.0,2006-12-16 17:24:00
1,16/12/2006,17:25:00,5.360,0.436,233.63,23.0,0.0,1.0,16.0,2006-12-16 17:25:00
2,16/12/2006,17:26:00,5.374,0.498,233.29,23.0,0.0,2.0,17.0,2006-12-16 17:26:00
3,16/12/2006,17:27:00,5.388,0.502,233.74,23.0,0.0,1.0,17.0,2006-12-16 17:27:00
4,16/12/2006,17:28:00,3.666,0.528,235.68,15.8,0.0,1.0,17.0,2006-12-16 17:28:00


In [ ]:
df["Global_active_power"] = pd.to_numeric(df["Global_active_power"], errors="coerce")
df.head()

,Date,Time,Global_active_power,Global_reactive_power,Voltage,Global_intensity,Sub_metering_1,Sub_metering_2,Sub_metering_3,Datetime
0,16/12/2006,17:24:00,4.216,0.418,234.84,18.4,0.0,1.0,17.0,2006-12-16 17:24:00
1,16/12/2006,17:25:00,5.360,0.436,233.63,23.0,0.0,1.0,16.0,2006-12-16 17:25:00
2,16/12/2006,17:26:00,5.374,0.498,233.29,23.0,0.0,2.0,17.0,2006-12-16 17:26:00
3,16/12/2006,17:27:00,5.388,0.502,233.74,23.0,0.0,1.0,17.0,2006-12-16 17:27:00
4,16/12/2006,17:28:00,3.666,0.528,235.68,15.8,0.0,1.0,17.0,2006-12-16 17:28:00


In [ ]:
df = df[["Datetime", "Global_active_power"]]
df.head()

,Datetime,Global_active_power
0,2006-12-16 17:24:00,4.216
1,2006-12-16 17:25:00,5.360
2,2006-12-16 17:26:00,5.374
3,2006-12-16 17:27:00,5.388
4,2006-12-16 17:28:00,3.666


In [ ]:
df = df.set_index("Datetime")
df = df.sort_index()
df["Global_active_power"] = df["Global_active_power"].interpolate(method="time")
df = df.dropna()
df.head()

,Global_active_power
Datetime,
2006-12-16 17:24:00,4.216
2006-12-16 17:25:00,5.360
2006-12-16 17:26:00,5.374
2006-12-16 17:27:00,5.388
2006-12-16 17:28:00,3.666


In [ ]:
df = df.resample("30min").mean()
df = df.dropna()
print("Resampled Shape:")
print(df.shape)
df.head()

Resampled Shape:
(69177, 1)


,Global_active_power
Datetime,
2006-12-16 17:00:00,4.587333
2006-12-16 17:30:00,4.150000
2006-12-16 18:00:00,3.944800
2006-12-16 18:30:00,3.319600
2006-12-16 19:00:00,3.464400


In [ ]:
print(df.info())

print("First Date:")
print(df.index.min())

print("Last Date:")
print(df.index.max())

print("Number of Rows:")
print(len(df))

<class 'pandas.core.frame.DataFrame'>
DatetimeIndex: 69177 entries, 2006-12-16 17:00:00 to 2010-11-26 21:00:00
Freq: 30min
Data columns (total 1 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   Global_active_power  69177 non-null  float64
dtypes: float64(1)
memory usage: 1.1 MB
None
First Date:
2006-12-16 17:00:00
Last Date:
2010-11-26 21:00:00
Number of Rows:
69177


In [ ]:
data = df["Global_active_power"].values.reshape(-1, 1)
print("Data Shape:")
print(data.shape)

Data Shape:
(69177, 1)


In [ ]:
train_size = int(len(data) * 0.80)
train_data = data[:train_size]
test_data = data[train_size:]
print("Training samples:", len(train_data))
print("Testing samples:", len(test_data))

Training samples: 55341
Testing samples: 13836


In [ ]:
scaler = MinMaxScaler(feature_range=(0, 1))
train_scaled = scaler.fit_transform(train_data)
test_scaled = scaler.transform(test_data)

In [ ]:
with open("scaler.pkl", "wb") as file:
    pickle.dump(scaler, file)

In [ ]:
SEQUENCE_LENGTH = 60

def create_sequences(data, sequence_length):
    X = []
    y = []
    for i in range(sequence_length, len(data)):
        X.append(data[i - sequence_length : i])
        y.append(data[i])
    return np.array(X), np.array(y)

In [ ]:
X_train, y_train = create_sequences(train_scaled, SEQUENCE_LENGTH)
print("X_train shape:", X_train.shape)
print("y_train shape:", y_train.shape)

X_train shape: (55281, 60, 1)
y_train shape: (55281, 1)


In [ ]:
combined_data = np.concatenate((train_scaled[-SEQUENCE_LENGTH:], test_scaled), axis=0)
X_test, y_test = create_sequences(combined_data, SEQUENCE_LENGTH)
print("X_test shape:", X_test.shape)
print("y_test shape:", y_test.shape)

X_test shape: (13836, 60, 1)
y_test shape: (13836, 1)


In [ ]:
model = Sequential(
    [
        LSTM(64, return_sequences=True, input_shape=(SEQUENCE_LENGTH, 1)),
        LSTM(32),
        Dense(1),
    ]
)

model.summary()

/usr/local/lib/python3.13/dist-packages/keras/src/layers/rnn/rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ lstm (LSTM)                     │ (None, 60, 64)         │        16,896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_1 (LSTM)                   │ (None, 32)             │        12,416 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 1)              │            33 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 29,345 (114.63 KB)

 Trainable params: 29,345 (114.63 KB)

 Non-trainable params: 0 (0.00 B)

In [ ]:
model.compile(optimizer="adam", loss="mean_squared_error")

In [ ]:
early_stopping = EarlyStopping(
    monitor="val_loss", patience=5, restore_best_weights=True
)

In [ ]:
history = model.fit(
    X_train,
    y_train,
    epochs=30,
    batch_size=64,
    validation_split=0.1,
    callbacks=[early_stopping],
    verbose=1,
)

Epoch 1/30
778/778 ━━━━━━━━━━━━━━━━━━━━ 11s 10ms/step - loss: 0.0071 - val_loss: 0.0065
Epoch 2/30
778/778 ━━━━━━━━━━━━━━━━━━━━ 8s 10ms/step - loss: 0.0057 - val_loss: 0.0062
Epoch 3/30
778/778 ━━━━━━━━━━━━━━━━━━━━ 7s 9ms/step - loss: 0.0056 - val_loss: 0.0062
Epoch 4/30
778/778 ━━━━━━━━━━━━━━━━━━━━ 7s 9ms/step - loss: 0.0055 - val_loss: 0.0060
Epoch 5/30
778/778 ━━━━━━━━━━━━━━━━━━━━ 8s 10ms/step - loss: 0.0055 - val_loss: 0.0060
Epoch 6/30
778/778 ━━━━━━━━━━━━━━━━━━━━ 7s 9ms/step - loss: 0.0055 - val_loss: 0.0060
Epoch 7/30
778/778 ━━━━━━━━━━━━━━━━━━━━ 8s 10ms/step - loss: 0.0055 - val_loss: 0.0060
Epoch 8/30
778/778 ━━━━━━━━━━━━━━━━━━━━ 7s 9ms/step - loss: 0.0055 - val_loss: 0.0060
Epoch 9/30
778/778 ━━━━━━━━━━━━━━━━━━━━ 8s 10ms/step - loss: 0.0054 - val_loss: 0.0059
Epoch 10/30
778/778 ━━━━━━━━━━━━━━━━━━━━ 7s 9ms/step - loss: 0.0054 - val_loss: 0.0059
Epoch 11/30
778/778 ━━━━━━━━━━━━━━━━━━━━ 7s 9ms/step - loss: 0.0054 - val_loss: 0.0059
Epoch 12/30
778/778 ━━━━━━━━━━━━━━━━━━━━ 8s 10

In [ ]:
test_loss = model.evaluate(X_test, y_test, verbose=1)
print("Test Loss:", test_loss)

433/433 ━━━━━━━━━━━━━━━━━━━━ 2s 4ms/step - loss: 0.0040
Test Loss: 0.003999180626124144


In [ ]:
predictions_scaled = model.predict(X_test)
print("Prediction shape:")
print(predictions_scaled.shape)

433/433 ━━━━━━━━━━━━━━━━━━━━ 2s 5ms/step
Prediction shape:
(13836, 1)


In [ ]:
predictions = scaler.inverse_transform(predictions_scaled)
actual_values = scaler.inverse_transform(y_test)

In [ ]:
mae = mean_absolute_error(actual_values, predictions)
print(f"MAE: {mae:.4f} kW")

MAE: 0.3256 kW


In [ ]:
mse = mean_squared_error(actual_values, predictions)
print(f"MSE: {mse:.4f}")

MSE: 0.2426


In [ ]:
rmse = np.sqrt(mse)
print(f"RMSE: {rmse:.4f} kW")

RMSE: 0.4925 kW


In [ ]:
results = pd.DataFrame(
    {
        "Actual Power (kW)": actual_values.flatten(),
        "Predicted Power (kW)": predictions.flatten(),
    }
)
results["Absolute Error (kW)"] = abs(
    results["Actual Power (kW)"] - results["Predicted Power (kW)"]
)
results.head(20)

,Actual Power (kW),Predicted Power (kW),Absolute Error (kW)
0,0.398933,0.434607,0.035674
1,0.349467,0.533651,0.184184
2,0.606000,0.524264,0.081736
3,0.669467,0.828398,0.158931
4,0.641067,0.939390,0.298324
5,1.078067,1.011152,0.066915
6,1.794667,1.464250,0.330417
7,2.067133,2.013834,0.053299
8,2.572667,2.160175,0.412492
9,2.379333,2.456036,0.076703


In [ ]:
model.save("model.keras")